# Financial 10-K RAG Assistant

This notebook runs the same final **GPT-5.6 Sol + OpenAI text-embedding-3-large** pipeline as the local Streamlit application. It uses the included Alphabet, Amazon, and Microsoft 2025 Form 10-K PDFs, requires no Google Drive connection, and does not use ngrok or Cloudflare tunnels.

API calls are billable. Never paste an API key into a saved notebook cell.

## 1. Load the GitHub repository

In [ ]:
from pathlib import Path
import os, shutil, subprocess, zipfile
from google.colab import files

PROJECT_ROOT = Path('/content/financial-10k-rag')
repo_url = input('GitHub repository URL (leave blank to upload the repository ZIP): ').strip()

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)

if repo_url:
    subprocess.run(['git', 'clone', '--depth', '1', repo_url, str(PROJECT_ROOT)], check=True)
else:
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(zip_names) != 1:
        raise ValueError('Upload exactly one ZIP archive of the financial-10k-rag repository.')
    archive = Path('/content') / zip_names[0]
    archive.write_bytes(uploaded[zip_names[0]])
    extract_dir = Path('/content/repository-upload')
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(extract_dir)
    candidates = [p for p in extract_dir.iterdir() if p.is_dir()]
    source = candidates[0] if len(candidates) == 1 else extract_dir
    shutil.move(str(source), str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)

## 2. Install dependencies

In [ ]:
subprocess.run([
    'python', '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)
], check=True)
print('Dependencies installed.')

## 3. Enter the OpenAI API key securely

In [ ]:
import getpass

os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ').strip()
if not os.environ['OPENAI_API_KEY']:
    raise ValueError('An OpenAI API key is required.')
print('API key received for this runtime only.')

## 4. Select the source PDFs

The repository already includes the three course filings. Set `USE_INCLUDED_PDFS = False` only if you want to upload all three replacements. Replacement filenames must identify Alphabet/Google, Amazon, and Microsoft.

In [ ]:
from financial_rag.document_processing import infer_company

USE_INCLUDED_PDFS = True
PDF_DIR = PROJECT_ROOT / 'data' / '10k'

if not USE_INCLUDED_PDFS:
    uploaded = files.upload()
    if len(uploaded) != 3:
        raise ValueError('Upload exactly three replacement PDFs.')
    replacement_dir = Path('/content/replacement-10k')
    replacement_dir.mkdir(parents=True, exist_ok=True)
    canonical = {
        'Alphabet/Google': 'Alphabet_10k_2025.pdf',
        'Amazon': 'Amazon_10k_2025.pdf',
        'Microsoft': 'Microsoft_10K_2025.pdf',
    }
    seen = set()
    for filename, data in uploaded.items():
        company = infer_company(filename)
        if company not in canonical or company in seen:
            raise ValueError(f'Invalid or duplicate company filename: {filename}')
        (replacement_dir / canonical[company]).write_bytes(data)
        seen.add(company)
    if seen != set(canonical):
        raise ValueError('One Alphabet, one Amazon, and one Microsoft PDF are required.')
    PDF_DIR = replacement_dir

pdf_paths = sorted(PDF_DIR.glob('*.pdf'))
if len(pdf_paths) != 3:
    raise ValueError(f'Expected three PDFs; found {len(pdf_paths)}.')
print('Source PDFs:')
for path in pdf_paths:
    print('-', path.name)

## 5. Build or load the RAG index

The final retrieval parameters are fixed. The model ID remains editable in case the API key cannot access GPT-5.6 Sol.

In [ ]:
from financial_rag import DEFAULT_LLM_MODEL, FinancialRAG, RAGConfig, validate_model

MODEL_ID = DEFAULT_LLM_MODEL
validate_model(MODEL_ID)
rag = FinancialRAG(RAGConfig(llm_model=MODEL_ID))
dimensions = rag.validate_embedding_credentials()
stats = rag.build_or_load(
    pdf_paths,
    cache_root=PROJECT_ROOT / 'cache' / 'faiss',
    rebuild=False,
)
print(f'Embedding dimensions: {dimensions}')
print('RAG ready:', stats)

## 6. Interactive question box

In [ ]:
import ipywidgets as widgets
from IPython.display import Markdown, clear_output, display

question_box = widgets.Textarea(
    value="What was Microsoft's Productivity and Business Processes segment revenue for fiscal year 2024?",
    placeholder='Ask a filing-grounded question...',
    description='Question:',
    layout=widgets.Layout(width='100%', height='110px'),
    style={'description_width': '80px'},
)
ask_button = widgets.Button(description='Ask', button_style='primary', icon='search')
output = widgets.Output()

def ask_question(_):
    question = question_box.value.strip()
    if not question:
        return
    with output:
        clear_output(wait=True)
        print('Retrieving evidence and generating the answer...')
        try:
            result = rag.answer(question)
            clear_output(wait=True)
            display(Markdown('### Answer'))
            display(Markdown(result['answer']))
            print(
                f"Model: {result['model']} | Strategy: {result['retrieval_strategy']} | "
                f"Latency: {result['latency_seconds']}s | Total tokens: {result['total_tokens']:,}"
            )
            display(Markdown('### Retrieved sources'))
            for source in result['sources']:
                print(
                    f"{source['rank']}. {source['company']} - {source['source_file']}, "
                    f"PDF page {source['page_number']} ({source['doc_type']})"
                )
        except Exception as exc:
            clear_output(wait=True)
            print(f'{type(exc).__name__}: {exc}')

ask_button.on_click(ask_question)
display(question_box, ask_button, output)

## 7. Optional direct Python call

For scripted use, replace the question below and rerun the cell.

In [ ]:
QUESTION = "What was Amazon's cash, cash equivalents, and restricted cash at the beginning of fiscal 2024?"
result = rag.answer(QUESTION)
display(Markdown(result['answer']))
print(f"Latency: {result['latency_seconds']}s | Strategy: {result['retrieval_strategy']}")